# NovaSight — Google Colab GPU Job Runner (Protocol v1.1)

This notebook executes approved experiment gates on a Google Colab GPU environment under Protocol v1.1.

**Safeguards**:
- No secrets or GitHub credentials stored or needed.
- Strictly read-only on the repository; does NOT push to Git from Colab.
- Enforces repository commit verification and dataset SHA-256 validation.
- Packs all generated artifacts and reports into a self-contained, hash-verified `RUN_BUNDLE.zip`.

---

## 1. Verify GPU Acceleration
Ensure runtime type is set to GPU (T4 / V100 / A100).

In [ ]:
!nvidia-smi

## 2. Clone Repository at Pinned Commit

In [ ]:
# Pin the approved Protocol v1.1 commit SHA here:
PINNED_COMMIT = "bce5403"  # Update to exact approved Gate 3 commit

!git clone https://github.com/VINYTHA1708/NovaSight.git
%cd NovaSight
!git checkout {PINNED_COMMIT}
!git status

## 3. Install Dependencies
Preserves Colab pre-installed CUDA PyTorch.

In [ ]:
!pip install -r requirements-colab.txt

## 4. Verify Dataset Integrity
Asserts that all 1,800 NEU-CLS images exist and match SHA-256 hashes bit-for-bit.

In [ ]:
!python scripts/check_dataset.py

## 5. Execute Approved Gate Jobs
Runs sequential jobs under `--gate` with strict commit assertion.

In [ ]:
# Set gate to run (e.g. 4 for rehearsal/selection, 5 for artifact build, 6 for evaluation)
GATE_NUM = 4
ROWS = "all"
OUT_DIR = "/content/out_bundle"

!python scripts/run_jobs.py \
    --gate {GATE_NUM} \
    --rows {ROWS} \
    --out {OUT_DIR} \
    --device cuda \
    --expect-commit {PINNED_COMMIT}

## 6. Package Bundle for Local Import
Zips the output bundle containing all artifacts, reports, and RUN_BUNDLE.json.

In [ ]:
%cd {OUT_DIR}
!zip -r /content/RUN_BUNDLE_gate{GATE_NUM}.zip .

from google.colab import files
files.download(f"/content/RUN_BUNDLE_gate{GATE_NUM}.zip")

## 7. Local Import Instructions
After downloading the zip archive on your local workstation:
1. Unzip the archive into a local temporary folder (e.g., `extracted_bundle/`).
2. Run the verified local importer:
   ```powershell
   .venv\Scripts\python.exe scripts\import_results.py extracted_bundle/RUN_BUNDLE.json
   ```
3. The importer verifies artifact hashes, checks commit ancestry, copies to `reports/v1.1/` and `artifacts/runs/v1.1/`, runs `verify_artifact.py`, and updates `reports/run_registry.csv`.